In [ ]:
import pandas as pd
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import src.config as config
import src.profile_model_registry as profile_model_registry
import src.profile_tracker as profile_tracker
import src.profile_trainer as profile_trainer

importlib.reload(config)
importlib.reload(profile_model_registry)
importlib.reload(profile_tracker)
importlib.reload(profile_trainer)

from src.profile_trainer import run_profile_training_experiment

print(profile_model_registry.get_profile_dense_models().keys())
print(profile_model_registry.get_profile_nan_friendly_models().keys())

In [ ]:
path = r"C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\15_min\df_BU_TotActPwr_Tech_Room.parquet"
df_imputed = pd.read_parquet(path)

In [ ]:
df_imputed.head()

In [ ]:
df_imputed.columns.tolist()

## Baseline + residulas

In [ ]:
from src.splitter import time_based_split, extract_xy
from src.baseline_features import (
    build_slot_baseline_table,
    apply_slot_baseline,
    add_residual_target,
)
from src.model_registry import get_dense_models
from src.metrics import evaluate_regression

In [ ]:
target_col = "BU_TotActPwr_Tech_Room"

train_df, val_df, test_df = time_based_split(
    df_imputed,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
)

print(train_df.shape, val_df.shape, test_df.shape)

In [ ]:
baseline_table = build_slot_baseline_table(
    train_df,
    target_col=target_col,
    use_median=True,
)

baseline_table.head()

In [ ]:
train_b = apply_slot_baseline(train_df, baseline_table=baseline_table, target_col=target_col)
val_b   = apply_slot_baseline(val_df,   baseline_table=baseline_table, target_col=target_col)
test_b  = apply_slot_baseline(test_df,  baseline_table=baseline_table, target_col=target_col)

In [ ]:
f"{target_col}_baseline"

In [ ]:
print(train_b[[target_col, f"{target_col}_baseline"]].head())

In [ ]:
train_b = add_residual_target(train_b, target_col=target_col)
val_b   = add_residual_target(val_b, target_col=target_col)
test_b  = add_residual_target(test_b, target_col=target_col)

In [ ]:
train_b[[target_col, f"{target_col}_baseline", f"{target_col}_residual"]].head()

In [ ]:
target_col = "BU_TotActPwr_Tech_Room"
residual_target_col = f"{target_col}_residual"
baseline_col = f"{target_col}_baseline"

feature_cols_residual = [
    c for c in train_b.columns
    if c not in [target_col, residual_target_col]
]

print("Number of residual features:", len(feature_cols_residual))
print(feature_cols_residual[:20])

In [ ]:
train_fit = train_b.dropna(subset=feature_cols_residual + [residual_target_col]).copy()
val_fit   = val_b.dropna(subset=feature_cols_residual + [residual_target_col]).copy()
test_fit  = test_b.dropna(subset=feature_cols_residual + [residual_target_col]).copy()

In [ ]:
X_train, y_train = extract_xy(train_fit, residual_target_col, feature_cols_residual)
X_val, y_val     = extract_xy(val_fit, residual_target_col, feature_cols_residual)
X_test, y_test   = extract_xy(test_fit, residual_target_col, feature_cols_residual)

In [ ]:
models = get_dense_models(random_state=42)
model = models["xgboost"]   # or "ridge" if you want simple first test

model.fit(X_train, y_train)

In [ ]:
test_residual_pred = model.predict(X_test)

test_final_pred = test_fit[baseline_col].values + test_residual_pred
test_final_true = test_fit[target_col].values

test_metrics = evaluate_regression(test_final_true, test_final_pred)
test_metrics

In [ ]:
print("train end:", train_df.index.max())
print("val start :", val_df.index.min())
print("val end   :", val_df.index.max())
print("test start:", test_df.index.min())

assert train_df.index.max() < val_df.index.min(), "Leakage risk: train overlaps val"
assert val_df.index.max() < test_df.index.min(), "Leakage risk: val overlaps test"

print("Chronological split check passed.")

In [ ]:
target_col = "BU_TotActPwr_Tech_Room"
residual_target_col = f"{target_col}_residual"
baseline_col = f"{target_col}_baseline"

forbidden = [target_col, residual_target_col]

bad_features = [c for c in feature_cols_residual if c in forbidden]
print("Forbidden features found:", bad_features)

assert target_col not in feature_cols_residual, "Leakage: raw target is in features"
assert residual_target_col not in feature_cols_residual, "Leakage: residual target is in features"

print("Feature leakage check passed.")

In [ ]:
print(baseline_table.head())
print(train_b[[target_col, baseline_col]].head())
print(val_b[[target_col, baseline_col]].head())
print(test_b[[target_col, baseline_col]].head())

In [ ]:
suspicious_equal_to_target = []
suspicious_equal_to_residual = []

for col in feature_cols_residual:
    if train_fit[col].equals(train_fit[target_col]):
        suspicious_equal_to_target.append(col)
    if train_fit[col].equals(train_fit[residual_target_col]):
        suspicious_equal_to_residual.append(col)

print("Exactly equal to raw target:", suspicious_equal_to_target)
print("Exactly equal to residual target:", suspicious_equal_to_residual)

In [ ]:
suspicious_patterns = ["lead", "future", "tplus", "next", "ahead"]
suspicious_cols = [
    c for c in feature_cols_residual
    if any(p in c.lower() for p in suspicious_patterns)
]

print("Possibly suspicious feature names:", suspicious_cols)

In [ ]:
from src.metrics import evaluate_regression

# Predict residuals
train_residual_pred = model.predict(X_train)
val_residual_pred   = model.predict(X_val)
test_residual_pred  = model.predict(X_test)

# Reconstruct final load forecasts
train_final_pred = train_fit[baseline_col].values + train_residual_pred
val_final_pred   = val_fit[baseline_col].values + val_residual_pred
test_final_pred  = test_fit[baseline_col].values + test_residual_pred

# True values
train_final_true = train_fit[target_col].values
val_final_true   = val_fit[target_col].values
test_final_true  = test_fit[target_col].values

# Metrics
train_metrics = evaluate_regression(train_final_true, train_final_pred)
val_metrics   = evaluate_regression(val_final_true, val_final_pred)
test_metrics  = evaluate_regression(test_final_true, test_final_pred)

train_metrics, val_metrics, test_metrics

In [ ]:

comparison_df = pd.DataFrame([
    {"split": "train", **train_metrics},
    {"split": "val", **val_metrics},
    {"split": "test", **test_metrics},
])

comparison_df

In [ ]:
baseline_test_pred = test_fit[baseline_col].values
baseline_test_true = test_fit[target_col].values

baseline_test_metrics = evaluate_regression(baseline_test_true, baseline_test_pred)
baseline_test_metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_actual_vs_pred(index, y_true, y_pred, title="Actual vs Predicted", start=None, end=None):
    df_plot = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
    }, index=pd.to_datetime(index)).sort_index()

    if start is not None:
        df_plot = df_plot[df_plot.index >= pd.to_datetime(start)]
    if end is not None:
        df_plot = df_plot[df_plot.index <= pd.to_datetime(end)]

    plt.figure(figsize=(14, 5))
    plt.plot(df_plot.index, df_plot["y_true"], label="Actual")
    plt.plot(df_plot.index, df_plot["y_pred"], label="Predicted")
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel(target_col)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_actual_vs_pred(
    index=test_fit.index,
    y_true=test_final_true,
    y_pred=test_final_pred,
    title="Baseline + Residual: Test Set Actual vs Predicted"
)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(test_final_true, test_final_pred, alpha=0.4)
min_val = min(test_final_true.min(), test_final_pred.min())
max_val = max(test_final_true.max(), test_final_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Test Set: Actual vs Predicted Scatter")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
residuals_test = test_final_true - test_final_pred

plt.figure(figsize=(14, 4))
plt.plot(test_fit.index, residuals_test)
plt.axhline(0, linestyle="--")
plt.title("Test Residuals Over Time")
plt.xlabel("Time")
plt.ylabel("Residual")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()